# Tests — utils_ai_queue

In [ ]:
import os
from fastsql import Database
from fh_saas.utils_ai_queue import AIQueueManager

DB_PATH = 'test_ai_queue.db'
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

db = Database(f'sqlite:///{DB_PATH}')
qm = AIQueueManager(db)
print('✅ queue manager initialized')

In [ ]:
job_id = qm.enqueue(tenant_id='tenant_a', job_type='doc_format', payload={'text': 'hello'})
job = qm.get_job(job_id)
assert job is not None
assert job.status == 'queued'
print('✅ enqueue creates queued job')

In [ ]:
acquired = qm.acquire_next_job(worker_id='worker_1', lease_seconds=20)
assert acquired is not None
assert acquired.id == job_id
assert acquired.status == 'processing'

none_for_other = qm.acquire_next_job(worker_id='worker_2', lease_seconds=20)
assert none_for_other is None
print('✅ lease prevents double-acquire')

In [ ]:
ok = qm.heartbeat(job_id, worker_id='worker_1', lease_seconds=30)
assert ok is True
job = qm.get_job(job_id)
assert job.lease_until is not None
print('✅ heartbeat extends lease')

In [ ]:
ok = qm.complete(job_id, worker_id='worker_1', result={'text': 'done'})
assert ok is True
job = qm.get_job(job_id)
assert job.status == 'done'
print('✅ complete marks done')

In [ ]:
job2 = qm.enqueue(tenant_id='tenant_a', job_type='unstable', payload={'x': 1}, max_retries=2)


def boom(**kwargs):
    raise RuntimeError('boom')

qm.execute_once(worker_id='worker_retry', task_func=boom)
state1 = qm.get_job(job2)
assert state1.status in ('queued', 'failed')

qm.execute_once(worker_id='worker_retry', task_func=boom)
state2 = qm.get_job(job2)
assert state2.status == 'failed'
print('✅ retry then fail terminally')

In [ ]:
job3 = qm.enqueue(tenant_id='tenant_b', job_type='cancel_me', payload={'x': 2})
qm.request_cancel(job3)
qm.execute_once(worker_id='worker_cancel', task_func=lambda **k: {'ok': True})
state3 = qm.get_job(job3)
assert state3.status == 'canceled'
print('✅ cancel requested is respected')

In [ ]:
db.conn.close()
try:
    db.engine.dispose()
except Exception:
    pass
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
print('✅ cleanup done')